# Sparse J-space decomposition on Gemma 4 E4B — gradient pursuit

Decomposes held-out evaluation activations into sparse nonnegative
combinations of J-lens vectors (rows of `W_U @ J_l`) using gradient pursuit,
against the **frozen 100-prompt pilot lens**
(`runs/pilot_20260715T200437612150_311fd108c23a/artifacts/lens.pt`) — the
lens is verified by file fingerprint and **never refitted**. Also runs the
improved named-control evaluation (`jlens/evaluation.py`) on the categorized
held-out prompt set v2.

Method documentation: `docs/jspace_decomposition.md`. Pilot findings and
interpretation boundaries: `docs/pilot_report.md`.

**Boundaries:** gradient pursuit identifies the *local cone* approximating
one activation; recurring-cone aggregation is a separate transparent stage;
`jlens/ignition.py` outputs are *candidate* diagnostics, not detected
ignition. The paper reports J-space components capture ≤ ~10% of activation
variance — high residuals on real activations are expected.


## 0. Colab setup (skip if running locally)

In [ ]:
# 0. Colab bootstrap: clone/update the private repo from a fresh runtime.
# No-op outside Colab. Never loads Gemma; only touches git/pip.
import base64
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("Not running in Colab — skipping bootstrap; using the local checkout.")
else:
    CHECKOUT_DIR = Path("/content/jacobian-lens-gemma")
    REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
    BRANCH = "jspace-gradient-pursuit"

    def _normalize(url: str) -> str:
        return url.strip().removesuffix(".git").removesuffix("/")

    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN secret not found or not accessible. In Colab: open "
            "the key icon (Secrets) in the left sidebar, add a secret named "
            "GITHUB_TOKEN containing a fine-grained GitHub token with "
            "read-only access to MechInterpreter/jacobian-lens-gemma, and "
            "enable notebook access for it. Then re-run this cell."
        )

    # Auth via a per-invocation extraHeader override: lives only in argv,
    # never written to .git/config, the remote URL, or notebook output.
    _token_b64 = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    _auth_header = f"AUTHORIZATION: basic {_token_b64}"
    _auth_args = ["-c", f"http.https://github.com/.extraHeader={_auth_header}"]

    def _run(args, *, auth=False, check=True):
        cmd = ["git", *(_auth_args if auth else []), *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            safe_stderr = result.stderr.replace(GITHUB_TOKEN, "***")
            raise RuntimeError(f"git {' '.join(args)} failed:\n{safe_stderr}")
        return result.stdout.strip()

    if not CHECKOUT_DIR.exists():
        print(f"Cloning {REPO_URL} ({BRANCH}) into {CHECKOUT_DIR} ...")
        _run(["clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)], auth=True)
    else:
        if not (CHECKOUT_DIR / ".git").exists():
            raise RuntimeError(
                f"{CHECKOUT_DIR} exists but is not a git checkout; refusing to "
                "touch it. Remove or rename it manually, then re-run this cell."
            )
        existing_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
        if _normalize(existing_url) != _normalize(REPO_URL):
            raise RuntimeError(
                f"{CHECKOUT_DIR} is checked out from {existing_url!r}, not "
                f"{REPO_URL!r}; refusing to touch an unexpected repository. "
                "Remove or rename it manually, then re-run this cell."
            )
        print(f"Updating existing checkout at {CHECKOUT_DIR} ...")
        _run(["-C", str(CHECKOUT_DIR), "fetch", "origin"], auth=True)
        _run(["-C", str(CHECKOUT_DIR), "checkout", BRANCH])
        _run(["-C", str(CHECKOUT_DIR), "merge", "--ff-only", f"origin/{BRANCH}"])

    final_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
    if "@" in final_url or GITHUB_TOKEN in final_url:
        raise RuntimeError("origin URL unexpectedly contains credentials; aborting.")

    os.chdir(CHECKOUT_DIR)
    if str(CHECKOUT_DIR) not in sys.path:
        sys.path.insert(0, str(CHECKOUT_DIR))

    print("Installing the 'gemma' extra ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gemma]"],
        check=True,
    )

    branch_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
    sha_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "HEAD"])
    print(f"checked out: {branch_now} @ {sha_now}")
    assert (CHECKOUT_DIR / "jlens").is_dir(), f"{CHECKOUT_DIR}/jlens not found"


In [ ]:
# 1. Execution gates. Model loading and CUDA are opt-in; outside Colab the
# defaults keep everything in the light (no-download) path so the notebook's
# structure can be validated locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("JLENS_ALLOW_GEMMA", "1" if IN_COLAB else "0")
os.environ.setdefault("JLENS_DEVICE_MAP", "cuda" if IN_COLAB else "")
# Resuming a specific prior run is an explicit opt-in (its exact RUN_DIR):
# os.environ["JSPACE_RESUME_RUN_DIR"] = ".../runs/jspace_..."
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None
print(f"IN_COLAB={IN_COLAB}  ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")


In [ ]:
# 2. Environment and provenance (no model load).
import json
import pathlib
import time

import torch

from jlens.metadata import environment_manifest

ENV = environment_manifest()
print(json.dumps(ENV, indent=2))


## Persisting outputs to Google Drive

Same pattern as the pilot notebook: outputs go to a **new unique run
directory** under `.../jacobian-lens-gemma/runs/` on Drive. The pilot run
directory is read (lens + metadata) but **never written to**. Model weights
and HF caches stay in the ephemeral `/content` filesystem.

In [ ]:
# 3. Google Drive persistence (Colab only). No-op outside Colab.
from pathlib import Path

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            f"failed to mount Google Drive at /content/drive: {exc}. Approve "
            "the Drive authorization prompt when it appears, then re-run this cell."
        ) from exc

    DRIVE_MOUNT = Path("/content/drive")
    if not DRIVE_MOUNT.is_dir():
        raise RuntimeError("Drive did not mount successfully.")

    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
    RUNS_ROOT = PERSIST_ROOT / "runs"
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"RUNS_ROOT = {RUNS_ROOT}")
else:
    PERSIST_ROOT = None
    RUNS_ROOT = Path("runs")  # local checkout: read the archived pilot run
    print("Not in Colab — using the local runs/ directory (read) and "
          "artifacts/jspace (write).")


In [ ]:
# 4. Load and validate the decomposition configuration; create (or resume,
# explicitly and fingerprint-validated) the run directory.
from datetime import datetime, timezone

from jlens.metadata import config_fingerprint, load_jspace_config

CONFIG_PATH = "configs/gemma_jspace_pursuit.yaml"
CONFIG = load_jspace_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(CONFIG)
DEC = CONFIG["decomposition"]
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")
print(f"decomposition layers={DEC['layers']}  k_values={DEC['k_values']}")

RESUME_RUN_DIR_ENV = os.environ.get("JSPACE_RESUME_RUN_DIR") or None
if IN_COLAB:
    if RESUME_RUN_DIR_ENV is not None:
        RUN_DIR = pathlib.Path(RESUME_RUN_DIR_ENV)
        prior_path = RUN_DIR / "run_metadata.json"
        if not prior_path.is_file():
            raise RuntimeError(
                f"JSPACE_RESUME_RUN_DIR={RUN_DIR} has no run_metadata.json; "
                "point it at a previously started jspace run directory"
            )
        prior = json.load(open(prior_path, encoding="utf-8"))
        if prior.get("config_fingerprint") != FINGERPRINT:
            raise RuntimeError(
                f"refusing to resume {RUN_DIR}: recorded fingerprint "
                f"{prior.get('config_fingerprint')!r} != active {FINGERPRINT!r}"
            )
        RUN_ID = RUN_DIR.name
        print(f"resuming explicit run: {RUN_DIR} (fingerprint verified)")
    else:
        RUN_ID = (f"jspace_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}_"
                  f"{FINGERPRINT.removeprefix('sha256:')[:12]}")
        RUN_DIR = RUNS_ROOT / RUN_ID
        RUN_DIR.mkdir(parents=True, exist_ok=False)  # must not already exist
    OUTPUT_DIR = RUN_DIR / "artifacts"
else:
    RUN_ID = None
    RUN_DIR = None
    OUTPUT_DIR = pathlib.Path(CONFIG["paths"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONES_DIR = OUTPUT_DIR / "cones"
CONES_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_DIR = {RUN_DIR}\nOUTPUT_DIR = {OUTPUT_DIR}")


In [ ]:
# 5. Lightweight validation suite (CPU, mocks, no network) — must pass
# before any real-model work, exactly as in the fitting notebook.
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("local test suite failed; fix before running the model")


In [ ]:
# 6. Locate and verify the FROZEN pilot lens. Read-only: fingerprint, model
# revision (from the pilot run's own metadata), fitted layers, dimensions,
# and finite values. Never refitted, never overwritten.
from jlens.lens import JacobianLens
from jlens.metadata import file_sha256

LENS_CFG = CONFIG["lens"]
PILOT_RUN_DIR = RUNS_ROOT / LENS_CFG["run_dir_name"]
LENS_PATH = PILOT_RUN_DIR / LENS_CFG["artifact_relpath"]
if not LENS_PATH.is_file():
    raise RuntimeError(
        f"pilot lens not found at {LENS_PATH}. In Colab, RUNS_ROOT must be the "
        "Drive runs/ directory containing the completed pilot run "
        f"{LENS_CFG['run_dir_name']!r}."
    )

lens_sha = file_sha256(str(LENS_PATH))
print(f"lens file: {LENS_PATH}\nsha256:    {lens_sha}")
if LENS_CFG["expect_file_sha256"] and lens_sha != LENS_CFG["expect_file_sha256"]:
    raise RuntimeError(
        f"lens fingerprint mismatch: expected {LENS_CFG['expect_file_sha256']}, "
        f"got {lens_sha} — refusing to decompose against an unexpected lens"
    )

LENS = JacobianLens.load(str(LENS_PATH))
assert LENS.source_layers == LENS_CFG["expect_source_layers"], LENS.source_layers
assert LENS.n_prompts == LENS_CFG["expect_n_prompts"], LENS.n_prompts
assert LENS.d_model == CONFIG["model"]["expect_d_model"], LENS.d_model
for layer, J in LENS.jacobians.items():
    assert J.shape == (LENS.d_model, LENS.d_model), (layer, J.shape)
    assert torch.isfinite(J).all(), f"non-finite J at layer {layer}"

pilot_meta_path = PILOT_RUN_DIR / "run_metadata.json"
PILOT_META = json.load(open(pilot_meta_path, encoding="utf-8"))
assert PILOT_META["model_revision"] == LENS_CFG["expect_model_revision"], (
    PILOT_META["model_revision"]
)
LENS_VERIFICATION = {
    "lens_path": str(LENS_PATH),
    "file_sha256": lens_sha,
    "source_layers": LENS.source_layers,
    "n_prompts": LENS.n_prompts,
    "d_model": LENS.d_model,
    "all_finite": True,
    "pilot_run_id": PILOT_META["run_id"],
    "pilot_model_revision": PILOT_META["model_revision"],
    "pilot_config_fingerprint": PILOT_META["config_fingerprint"],
}
print(json.dumps(LENS_VERIFICATION, indent=2))
print(f"verified: {LENS} @ pilot revision {PILOT_META['model_revision']}")


In [ ]:
# 7. Load the SAME immutable Gemma revision the lens was fitted on (gated).
# With model loading disabled this cell prints the skip reason and the rest
# of the notebook stays in the light path.
MODEL = None
LOAD_INFO = None
if not ALLOW_MODEL_LOAD:
    print("JLENS_ALLOW_GEMMA != 1 — model loading disabled; light path only.")
else:
    from jlens.gemma4 import load_gemma4, resolve_revision, verify_architecture

    pinned = CONFIG["model"]["revision"]
    resolved = resolve_revision(CONFIG["model"]["repo_id"], pinned)
    assert resolved == LENS_CFG["expect_model_revision"], (
        f"resolved revision {resolved} != lens revision "
        f"{LENS_CFG['expect_model_revision']}"
    )
    print(f"{CONFIG['model']['repo_id']} pinned to {resolved}")

    _DTYPES = {"bfloat16": torch.bfloat16, "float32": torch.float32}
    MODEL, LOAD_INFO = load_gemma4(
        CONFIG["model"]["repo_id"],
        revision=resolved,
        dtype=_DTYPES[CONFIG["model"]["dtype"]],
        device_map=DEVICE_MAP,
        allow_model_load=True,
    )
    report = verify_architecture(
        MODEL,
        expect_n_layers=CONFIG["model"]["expect_n_layers"],
        expect_d_model=CONFIG["model"]["expect_d_model"],
        expect_vocab_size=CONFIG["model"]["expect_vocab_size"],
    )
    ARCH_REPORT = report.to_dict()
    print(f"architecture verified: {ARCH_REPORT['model_class']} "
          f"({ARCH_REPORT['n_layers']}L, d={ARCH_REPORT['d_model']}, "
          f"V={ARCH_REPORT['vocab_size']})")


In [ ]:
# 8. Capture residual activations for the held-out evaluation prompts
# (one forward per prompt; per-prompt progress prints). These exact
# residuals are what gets decomposed — no refitting anywhere.
from jlens.evaluation import capture_residuals, load_eval_prompts_v2
from jlens.metadata import prompt_hashes

DEC_LAYERS = DEC["layers"]
if MODEL is None:
    print("model not loaded — skipping capture (light path).")
    CAPTURE_META, RESIDUALS, MODEL_TOP1 = None, None, None
else:
    rows = load_eval_prompts_v2(CONFIG["eval"]["prompts_path"], MODEL.tokenizer)
    print(f"{len(rows)} eval prompts (plain+chat), layers {DEC_LAYERS}")
    CAPTURE_META = []           # one entry per (prompt, position)
    per_layer_chunks = {layer: [] for layer in DEC_LAYERS}
    t0 = time.perf_counter()
    for row_idx, row in enumerate(rows):
        residuals, model_logits, input_ids = capture_residuals(
            MODEL, row["text"], layers=DEC_LAYERS,
            positions=row["positions"],
            max_seq_len=CONFIG["eval"]["max_seq_len"],
        )
        row_hash = prompt_hashes([row["text"]])[0]
        top1 = model_logits.argmax(-1)
        seq_len = input_ids.shape[1]
        for i, pos in enumerate(row["positions"]):
            tok_id = int(input_ids[0, pos])
            CAPTURE_META.append({
                "slug": row["slug"], "category": row["category"],
                "format": row["format"], "position": int(pos),
                "prompt_hash": row_hash, "seq_len": int(seq_len),
                "input_token_id": tok_id,
                "input_token": MODEL.tokenizer.decode([tok_id]),
                "model_top1_id": int(top1[i]),
                "model_top1_token": MODEL.tokenizer.decode([int(top1[i])]),
            })
        for layer in DEC_LAYERS:
            per_layer_chunks[layer].append(residuals[layer].cpu())
        print(f"  [{row_idx + 1}/{len(rows)}] {row['slug']} ({row['format']}) "
              f"seq_len={seq_len}  elapsed={time.perf_counter() - t0:.0f}s",
              flush=True)
    RESIDUALS = {layer: torch.cat(chunks, dim=0)
                 for layer, chunks in per_layer_chunks.items()}
    MODEL_TOP1 = {(m["prompt_hash"], m["position"], m["format"]): m["model_top1_id"]
                  for m in CAPTURE_META}
    B = len(CAPTURE_META)
    print(f"captured {B} activations per layer "
          f"({[tuple(RESIDUALS[l].shape) for l in DEC_LAYERS][0]} each)")
    for layer in DEC_LAYERS:
        assert torch.isfinite(RESIDUALS[layer]).all(), f"non-finite h at L{layer}"


In [ ]:
# 9. Improved held-out evaluation: named controls with recorded provenance
# and aggregate statistics (median rank, MRR, hit rates; plain vs chat and
# per-category kept separate). Replaces the old single wrong-layer control
# for this and future runs; the pilot artifacts are untouched.
if MODEL is None:
    print("model not loaded — skipping evaluation (light path).")
else:
    from jlens.evaluation import build_control_suite, evaluate_suite
    from jlens.gemma4 import softcap_disabled
    from jlens.metadata import write_metadata

    suite = build_control_suite(LENS, control_seed=CONFIG["eval"]["control_seed"])
    print("variants:", ", ".join(suite))
    t0 = time.perf_counter()
    with softcap_disabled(MODEL):  # paper convention; rankings unaffected
        EVAL_RESULTS = evaluate_suite(
            MODEL, suite, rows,
            layers=DEC_LAYERS,
            top_k=CONFIG["eval"]["top_k"],
            max_seq_len=CONFIG["eval"]["max_seq_len"],
        )
    print(f"evaluated {EVAL_RESULTS['n_prompts']} prompts in "
          f"{time.perf_counter() - t0:.0f}s")
    write_metadata(str(OUTPUT_DIR / "eval_v2_results.json"), {
        "config_fingerprint": FINGERPRINT,
        "lens_verification": LENS_VERIFICATION,
        "load_info": LOAD_INFO,
        "results": EVAL_RESULTS,
        "environment": ENV,
    })
    # Quick look: jlens vs strongest controls at the last two layers (plain).
    for layer in DEC_LAYERS[-2:]:
        agg = EVAL_RESULTS["aggregates"]["plain"][str(layer)]
        line = "  ".join(
            f"{name}: mr={agg[name]['median_rank']:.0f} "
            f"h@10={agg[name]['hit_rate@10']:.2f}"
            for name in ("jlens", "logit_lens", "permuted", "adjacent_layer",
                         "distant_layer")
        )
        print(f"L{layer}  {line}")


In [ ]:
# 10. Gradient pursuit: decompose every captured activation at each
# configured layer and k. Resume-safe: each (layer, k) writes its own
# cones_layer{L}_k{K}.json and completed files are skipped on re-run.
# Progress lines include elapsed time, examples done, current layer/k, and
# the artifact path.
if MODEL is None:
    print("model not loaded — skipping decomposition (light path).")
else:
    from jlens.cones import make_cone_record, save_cone_records
    from jlens.pursuit import JSpaceDictionary, PursuitSettings, gradient_pursuit

    RUN_PROVENANCE = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "config_fingerprint": FINGERPRINT,
        "lens_fingerprint": LENS_VERIFICATION["file_sha256"],
        "lens_path": LENS_VERIFICATION["lens_path"],
        "lens_n_prompts": LENS_VERIFICATION["n_prompts"],
        "model_revision": LOAD_INFO["model_revision"],
        "local_commit": ENV.get("local_commit"),
        "upstream_commit": ENV.get("upstream_commit"),
    }
    _ATOM_DTYPES = {"float32": torch.float32, "float16": torch.float16}
    atoms_dtype = _ATOM_DTYPES[DEC.get("atoms_dtype", "float32")]
    W_U = MODEL._lm_head.weight
    norm_weight = None
    if DEC["fold_final_norm_weight"]:
        # Gemma RMSNorm applies (1 + weight); pass the effective multiplier.
        norm_weight = 1.0 + MODEL._final_norm.weight.detach().float()

    t_start = time.perf_counter()
    n_units = len(DEC_LAYERS) * len(DEC["k_values"])
    unit = 0
    for layer in DEC_LAYERS:
        t_dict = time.perf_counter()
        dictionary = JSpaceDictionary.from_lens(
            LENS, layer, W_U,
            final_norm_weight=norm_weight,
            device=W_U.device, dtype=atoms_dtype,
        )
        print(f"[L{layer}] dictionary {dictionary.n_atoms}x{dictionary.d_model} "
              f"built in {time.perf_counter() - t_dict:.1f}s", flush=True)
        for k in DEC["k_values"]:
            unit += 1
            out_path = CONES_DIR / f"cones_layer{layer:02d}_k{k:02d}.json"
            if out_path.exists():
                print(f"[L{layer} k={k}] exists — resume skip: {out_path}",
                      flush=True)
                continue
            settings = PursuitSettings(
                k=k,
                normalize_atoms=DEC["normalize_atoms"],
                refine_steps=DEC["refine_steps"],
                tol_relative_residual=float(DEC["tol_relative_residual"]),
                correlation_chunk_size=DEC["correlation_chunk_size"],
            )
            t_unit = time.perf_counter()
            result = gradient_pursuit(RESIDUALS[layer], dictionary, settings)
            pursuit_records = result.to_records()
            cone_records = []
            for meta, record in zip(CAPTURE_META, pursuit_records, strict=True):
                labels = [MODEL.tokenizer.decode([i]) for i in record["token_ids"]]
                cone_records.append(make_cone_record(
                    record,
                    decoded_labels=labels,
                    layer=layer,
                    position=meta["position"],
                    input_token_id=meta["input_token_id"],
                    input_token=meta["input_token"],
                    prompt_hash=meta["prompt_hash"],
                    prompt_slug=meta["slug"],
                    prompt_format=meta["format"],
                    run_provenance={**RUN_PROVENANCE,
                                    "category": meta["category"],
                                    "model_top1_id": meta["model_top1_id"],
                                    "model_top1_token": meta["model_top1_token"]},
                ))
            tmp_path = out_path.with_suffix(".json.tmp")
            save_cone_records(cone_records, str(tmp_path))
            os.replace(tmp_path, out_path)
            mean_expl = sum(
                r["reconstruction"]["explained_fraction"] for r in cone_records
            ) / len(cone_records)
            print(f"[{unit}/{n_units}] L{layer} k={k}: "
                  f"{len(cone_records)} activations in "
                  f"{time.perf_counter() - t_unit:.1f}s  "
                  f"mean_explained={mean_expl:.3f}  "
                  f"elapsed={time.perf_counter() - t_start:.0f}s  -> {out_path}",
                  flush=True)
        del dictionary
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print("decomposition complete")


In [ ]:
# 11. Trajectories, recurring cone signatures, and candidate-ignition
# diagnostics (explicitly labeled; every signal separately visible; the
# heuristic composite stays disabled).
if MODEL is None:
    print("model not loaded — skipping trajectory analysis (light path).")
else:
    from collections import defaultdict

    from jlens.cones import (
        cone_trajectory, load_cone_records, recurring_signatures,
    )
    from jlens.ignition import candidate_ignition_signals, export_transition_records

    for k in DEC["k_values"]:
        all_records = []
        for layer in DEC_LAYERS:
            all_records.extend(load_cone_records(
                str(CONES_DIR / f"cones_layer{layer:02d}_k{k:02d}.json")
            ))
        by_traj = defaultdict(list)
        for record in all_records:
            by_traj[(record["prompt_hash"], record["position"],
                     record["format"])].append(record)

        transitions_out, ignition_out = [], []
        for key, records in sorted(by_traj.items(), key=str):
            transitions = cone_trajectory(records)
            transitions_out.extend(transitions)
            by_layer = {r["layer"]: r for r in records}
            ignition_out.extend(candidate_ignition_signals(
                transitions, by_layer,
                model_top1_id=MODEL_TOP1.get(key),
            ))
        with open(OUTPUT_DIR / f"trajectories_k{k:02d}.json", "w",
                  encoding="utf-8") as fh:
            json.dump(transitions_out, fh, indent=2, ensure_ascii=False)
        export_transition_records(
            ignition_out, str(OUTPUT_DIR / f"ignition_candidates_k{k:02d}.json"))
        recur = recurring_signatures(all_records)
        with open(OUTPUT_DIR / f"recurring_signatures_k{k:02d}.json", "w",
                  encoding="utf-8") as fh:
            json.dump(recur, fh, indent=2, ensure_ascii=False)
        print(f"k={k}: {len(transitions_out)} transitions, "
              f"{len(ignition_out)} candidate records, "
              f"{sum(1 for r in recur if r['count'] > 1)} recurring signatures",
              flush=True)


In [ ]:
# 12. Run manifest + human-readable summary into the run directory.
if MODEL is None:
    print("model not loaded — nothing to summarize (light path).")
else:
    from jlens.metadata import write_metadata

    manifest = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "mode": "jspace_pursuit",
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "lens_verification": LENS_VERIFICATION,
        "load_info": LOAD_INFO,
        "architecture_report": ARCH_REPORT,
        "n_activations_per_layer": len(CAPTURE_META),
        "capture_meta": CAPTURE_META,
        "decomposition_units": sorted(p.name for p in CONES_DIR.glob("*.json")),
        "environment": ENV,
        "notes": (
            "Frozen pilot lens; no refitting. Cone signatures are "
            "deterministic bookkeeping, not concept claims. Ignition outputs "
            "are candidate diagnostics only."
        ),
    }
    write_metadata(str((RUN_DIR or OUTPUT_DIR) / "run_metadata.json"), manifest)

    lines = [
        f"# Run {RUN_ID or OUTPUT_DIR}",
        "",
        f"- mode: jspace_pursuit (gradient pursuit on the frozen pilot lens)",
        f"- lens: {LENS_VERIFICATION['lens_path']} ({LENS_VERIFICATION['file_sha256']})",
        f"- model: {LOAD_INFO['model_repo_id']} @ {LOAD_INFO['model_revision']}",
        f"- layers: {DEC_LAYERS}; k values: {DEC['k_values']}",
        f"- activations decomposed per (layer, k): {len(CAPTURE_META)}",
        f"- artifacts: eval_v2_results.json, cones/, trajectories_k*.json, "
        f"ignition_candidates_k*.json, recurring_signatures_k*.json",
        "",
        "Interpretation boundaries: see docs/jspace_decomposition.md and "
        "docs/pilot_report.md.",
    ]
    with open((RUN_DIR or OUTPUT_DIR) / "summary.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("\n".join(lines))


## Summary and limitations

- Every decomposition in this run is computed against the verified, frozen
  100-prompt pilot lens; nothing is refitted and the pilot run directory is
  never written to.
- The paper reports J-space components capture no more than ~10% of
  activation variance on its models; expect comparable or higher residuals
  here — reconstruction quality is a *measurement*, not a target.
- Candidate-ignition outputs are labeled diagnostics. Confounds that must
  be excluded before stronger claims: layer-dependent lens quality,
  tokenization granularity, WikiText fitting-corpus bias, and ordinary
  late-layer vocabulary alignment (the logit lens also converges at layers
  35–38).
- The held-out prompt set v2 is a small deterministic probe set across a
  few task categories, not a comprehensive benchmark; chat and plain
  formats are never aggregated together.